In [1]:
pip install numpy opencv-python scikit-image scikit-learn qiskit qiskit-machine-learning matplotlib seaborn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.1 MB/s eta 0:00:00


In [ ]:
import os
import glob
import shutil
from sklearn.model_selection import train_test_split

# Define base paths
BASE_DIR = "/content/drive/MyDrive/data_folder"  # Change to your Drive path, e.g., '/content/drive/MyDrive/data_folder'
EXTERNAL_NORMAL_DIR = "./path_to_external_normals"  # Update with your external normal folder path
OUTPUT_DIR = "./dataset_split"

# ---------------------------------------------------------
# 1. Gather All Normal Images
#    CHD images are now handled by cell 7jB2Unm0drsy.
# ---------------------------------------------------------
print("[*] Gathering all Normal images (23 local + external)...")

local_normals = glob.glob(os.path.join(BASE_DIR, "normal", "*.*"))
ext_normals = glob.glob(os.path.join(EXTERNAL_NORMAL_DIR, "*.*"))

all_normal_files = local_normals + ext_normals
print(f" -> Found total of {len(all_normal_files)} Normal images.")

# ---------------------------------------------------------
# 2. Stratified Proportional Split for Normal Class
# ---------------------------------------------------------
# Split 75% for Train, and 25% remaining for (Val + Test)
norm_train, norm_temp = train_test_split(
    all_normal_files, test_size=0.25, random_state=42, shuffle=True
)

# Split remaining 25% into Val (~15%) and Test (~10%) -> ratio 60:40 of the temp pool
norm_val, norm_test = train_test_split(
    norm_temp, test_size=0.40, random_state=42, shuffle=True
)

print(f" -> Normal Split Count: Train = {len(norm_train)} | Val = {len(norm_val)} | Test = {len(norm_test)}")

# ---------------------------------------------------------
# 3. Save Normal images into Structured Directories
#    CHD directories are assumed to be populated by cell 7jB2Unm0drsy
# ---------------------------------------------------------
normal_splits = {
    'train': norm_train,
    'val':   norm_val,
    'test':  norm_test
}

for split_name, file_list in normal_splits.items():
    target_dir = os.path.join(OUTPUT_DIR, split_name, "Normal")
    os.makedirs(target_dir, exist_ok=True) # Ensure 'Normal' category dir exists

    for idx, file_path in enumerate(file_list):
        ext = os.path.splitext(file_path)[1]
        new_filename = f"normal_{idx:03d}{ext}"
        shutil.copy(file_path, os.path.join(target_dir, new_filename))

print("\n[+] Normal dataset organization complete!")
print(f"[*] Structured Normal dataset saved to: {OUTPUT_DIR}")

[*] Gathering all Normal images (23 local + external)...
 -> Found total of 223 Normal images.
 -> Normal Split Count: Train = 167 | Val = 33 | Test = 23

[+] Normal dataset organization complete!
[*] Structured Normal dataset saved to: ./dataset_split


In [ ]:
def load_split_descriptors(split_dir):
    """
    Case-insensitive loader for 'Normal' (label 0) and 'CHD' (label 1) images.
    """
    categories = ['Normal', 'CHD']
    image_desc_map = []
    labels = []
    all_descriptors = []

    # Get actual subfolder names in directory
    existing_folders = os.listdir(split_dir)

    for label_idx, cat in enumerate(categories):
        # Match folder case-insensitively (e.g., 'chd', 'CHD', 'Normal', 'normal')
        target_folder = None
        for folder in existing_folders:
            if folder.lower() == cat.lower():
                target_folder = folder
                break

        if target_folder is None:
            raise FileNotFoundError(
                f"Could not find a folder matching '{cat}' in '{split_dir}'. "
                f"Available folders: {existing_folders}"
            )

        cat_path = os.path.join(split_dir, target_folder)

        # Match all common image extensions case-insensitively
        image_files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG', '*.BMP', '*.bmp']:
            image_files.extend(glob.glob(os.path.join(cat_path, ext)))

        print(f"[*] Found {len(image_files)} images for class '{cat}' ({target_folder}) in split '{split_dir}'")

        if len(image_files) == 0:
            raise ValueError(f"No valid image files found in '{cat_path}'.")

        for img_path in image_files:
            seg_img = preprocess_and_segment(img_path)
            if seg_img is not None:
                _, desc = sift.detectAndCompute(seg_img, None)
                image_desc_map.append(desc)
                labels.append(label_idx)
                if desc is not None:
                    all_descriptors.append(desc)

    return image_desc_map, np.array(labels), all_descriptors

In [ ]:
import os
import glob
import shutil
from sklearn.model_selection import train_test_split

# Base paths
SOURCE_DIR = "/content/drive/MyDrive/data_folder/chd"      # Path to your main data_folder
DEST_DIR = "/content/dataset_split"       # Path to your target dataset_split directory

# ---------------------------------------------------------
# Step 1: Gather all CHD images from source folders
# ---------------------------------------------------------
# List all subfolders where your CHD images are currently stored
chd_subfolders = ["train_chd", "val_chd", "test_chd", "fchd", "chd", "CHD"]

all_chd_files = []
for subfolder in chd_subfolders:
    folder_path = os.path.join(SOURCE_DIR, subfolder)
    if os.path.exists(folder_path):
        # Match all common image file extensions
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG', '*.bmp', '*.BMP']:
            all_chd_files.extend(glob.glob(os.path.join(folder_path, ext)))

# Remove duplicate file entries if any exist
all_chd_files = sorted(list(set(all_chd_files)))

print(f"[*] Total CHD images collected from source: {len(all_chd_files)}")

if len(all_chd_files) == 0:
    raise FileNotFoundError(
        f"No CHD images were found in {SOURCE_DIR}. "
        "Please verify your source folder path."
    )

# ---------------------------------------------------------
# Step 2: Split CHD images into Train (75%), Val (15%), Test (10%)
# ---------------------------------------------------------
# First split: 75% for Train, 25% remaining for (Val + Test)
chd_train, chd_temp = train_test_split(
    all_chd_files, test_size=0.25, random_state=42, shuffle=True
)

# Second split: Divide the 25% temp pool into Val (60% of temp = 15%) and Test (40% of temp = 10%)
chd_val, chd_test = train_test_split(
    chd_temp, test_size=0.40, random_state=42, shuffle=True
)

print(f"[*] Split Distribution:")
print(f" -> Train CHD : {len(chd_train)} images")
print(f" -> Val CHD   : {len(chd_val)} images")
print(f" -> Test CHD  : {len(chd_test)} images")

# ---------------------------------------------------------
# Step 3: Copy files to respective dataset_split folders
# ---------------------------------------------------------
splits = {
    'train': chd_train,
    'val': chd_val,
    'test': chd_test
}

for split_name, file_list in splits.items():
    # Target directory structure: dataset_split/{train|val|test}/CHD
    target_dir = os.path.join(DEST_DIR, split_name, "CHD")
    os.makedirs(target_dir, exist_ok=True)

    for idx, file_path in enumerate(file_list):
        ext = os.path.splitext(file_path)[1]
        new_filename = f"chd_{split_name}_{idx:03d}{ext}"
        destination_path = os.path.join(target_dir, new_filename)

        # Copy file to destination
        shutil.copy(file_path, destination_path)

print(f"\n[+] Successfully loaded all CHD images into '{DEST_DIR}' subdirectories!")

[*] Total CHD images collected from source: 189
[*] Split Distribution:
 -> Train CHD : 141 images
 -> Val CHD   : 28 images
 -> Test CHD  : 20 images

[+] Successfully loaded all CHD images into '/content/dataset_split' subdirectories!


In [ ]:
import os
import glob

train_dir = "./dataset_split/train"

print("Checking directory contents:")
for item in os.listdir(train_dir):
    item_path = os.path.join(train_dir, item)
    if os.path.isdir(item_path):
        files = os.listdir(item_path)
        print(f" -> Subfolder found: '{item}' | Contains {len(files)} items")

Checking directory contents:
 -> Subfolder found: 'CHD' | Contains 141 items
 -> Subfolder found: 'Normal' | Contains 167 items


In [ ]:
import os
print("Normal Train Images:", len(os.listdir('./dataset_split/train/Normal')))
print("CHD Train Images:", len(os.listdir('./dataset_split/train/CHD')))

Normal Train Images: 167
CHD Train Images: 141


In [ ]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt

from skimage.segmentation import chan_vese
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)
from sklearn.preprocessing import StandardScaler

# Qiskit for Quantum SVM Benchmark
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

# =========================================================
# STEP 1: Image Preprocessing & Chan-Vese Segmentation
# =========================================================
def preprocess_and_segment(img_path):
    """
    Applies Bilateral Filter, CLAHE enhancement, and Chan-Vese
    active contour segmentation to isolate key cardiac boundaries.
    """
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # 1. Bilateral filter removes noise while keeping structural edges sharp
    denoised = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)

    # 2. CLAHE enhances contrast for faint cardiac structures
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)

    # 3. Chan-Vese Active Contour Segmentation
    img_norm = enhanced / 255.0
    cv_mask = chan_vese(img_norm, mu=0.25, lambda1=1, lambda2=1, tol=1e-3, max_num_iter=100)

    return (cv_mask * 255).astype(np.uint8)

# =========================================================
# STEP 2: SIFT Feature Extraction & Loader
# =========================================================
sift = cv2.SIFT_create()

def load_split_descriptors(split_dir):
    """
    Loads images from 'Normal' (label 0) and 'CHD' (label 1) folders
    for a given split directory (train/val/test).
    """
    categories = ['Normal', 'CHD']
    image_desc_map = []
    labels = []
    all_descriptors = []

    for label_idx, cat in enumerate(categories):
        cat_path = os.path.join(split_dir, cat)
        image_files = glob.glob(os.path.join(cat_path, "*.[jJ][pP]*[gG]")) + \
                      glob.glob(os.path.join(cat_path, "*.png"))

        for img_path in image_files:
            seg_img = preprocess_and_segment(img_path)
            if seg_img is not None:
                _, desc = sift.detectAndCompute(seg_img, None)
                image_desc_map.append(desc)
                labels.append(label_idx)
                if desc is not None:
                    all_descriptors.append(desc)

    return image_desc_map, np.array(labels), all_descriptors

# =========================================================
# STEP 3: Bag of Visual Words (BoVW) Feature Vectorization
# =========================================================
def build_bovw_histograms(image_desc_map, kmeans_model, num_clusters):
    """
    Converts variable SIFT descriptors per image into a fixed-length K-dim histogram.
    """
    X_bovw = []
    for desc in image_desc_map:
        histogram = np.zeros(num_clusters)
        if desc is not None and len(desc) > 0:
            words = kmeans_model.predict(desc)
            for word in words:
                histogram[word] += 1
            # Normalize histogram frequency
            norm = np.linalg.norm(histogram)
            if norm > 0:
                histogram /= norm
        X_bovw.append(histogram)
    return np.array(X_bovw)

# =========================================================
# STEP 4: Evaluation Metrics Printout
# =========================================================
def evaluate_performance(y_true, y_pred, y_prob=None, model_name="Model"):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=1) # CHD = Positive (1)
    rec = recall_score(y_true, y_pred, pos_label=1)
    f1 = f1_score(y_true, y_pred, pos_label=1)

    print(f"\n==========================================")
    print(f"      {model_name.upper()} EVALUATION      ")
    print(f"==========================================")
    print(f"Accuracy  : {acc * 100:.2f}%")
    print(f"Precision : {prec * 100:.2f}%")
    print(f"Recall    : {rec * 100:.2f}%")
    print(f"F1-Score  : {f1 * 100:.2f}%")

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print(f"               Pred Normal  Pred CHD")
    print(f"Actual Normal  {cm[0][0]:<11} {cm[0][1]}")
    print(f"Actual CHD     {cm[1][0]:<11} {cm[1][1]}")

    if y_prob is not None:
        auc = roc_auc_score(y_true, y_prob)
        print(f"ROC-AUC   : {auc:.4f}")

# =========================================================
# STEP 5: Execution Pipeline
# =========================================================
def main():
    DATASET_SPLIT_DIR = "./dataset_split" # Point to your split directory
    K_VOCAB_SIZE = 50                     # Bag of Words vocabulary size

    print("[*] Extracting SIFT features from TRAIN set...")
    train_map, y_train, train_descriptors = load_split_descriptors(os.path.join(DATASET_SPLIT_DIR, "train"))

    # Check if any descriptors were extracted from the training set
    if not train_descriptors:
        print("\nCRITICAL ERROR: No SIFT descriptors were extracted from the training set.")
        print("This could be due to:")
        print("1. No images found in the training directories (Normal or CHD).")
        print("2. All images failed to load or preprocess (e.g., cv2.imread returned None).")
        print("3. SIFT failed to find any features in the processed images.")
        return # Exit if no descriptors to build vocabulary

    print("[*] Extracting SIFT features from VAL set...")
    val_map, y_val, _ = load_split_descriptors(os.path.join(DATASET_SPLIT_DIR, "val"))

    print("[*] Extracting SIFT features from TEST set...")
    test_map, y_test, _ = load_split_descriptors(os.path.join(DATASET_SPLIT_DIR, "test"))

    # 1. Build K-Means Visual Vocabulary strictly on TRAIN set to prevent data leakage
    # Filter out None descriptors before stacking
    valid_train_descriptors = [d for d in train_descriptors if d is not None and d.size > 0]

    if not valid_train_descriptors:
        print("\nCRITICAL ERROR: No valid SIFT descriptors available after filtering for K-Means vocabulary building.")
        print("This usually means `train_descriptors` was empty or contained only None/empty values.")
        return

    stacked_train_desc = np.vstack(valid_train_descriptors)
    print(f"\n[*] Training K-Means model on training descriptors (K = {K_VOCAB_SIZE})...")
    kmeans = KMeans(n_clusters=K_VOCAB_SIZE, random_state=42, n_init=10)
    kmeans.fit(stacked_train_desc)

    # 2. Transform splits into BoVW histograms
    X_train = build_bovw_histograms(train_map, kmeans, K_VOCAB_SIZE)
    X_val   = build_bovw_histograms(val_map, kmeans, K_VOCAB_SIZE)
    X_test  = build_bovw_histograms(test_map, kmeans, K_VOCAB_SIZE)

    # 3. Feature Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)

    # Check if y_train has at least two classes before training SVM
    unique_train_classes = np.unique(y_train)
    if len(unique_train_classes) < 2:
        print(f"\nCRITICAL ERROR: Only {len(unique_train_classes)} class(es) found in the training data (y_train).")
        print("SVM classification requires at least two classes. This usually indicates an issue with your dataset preparation.")
        print("Please ensure your 'train' split contains both 'Normal' (label 0) and 'CHD' (label 1) images.")
        print(f"Unique classes found in y_train: {unique_train_classes}")
        return # Exit the main function as SVM cannot be trained with only one class.

    # 4. Train Classical Support Vector Machine (SVM)
    print("\n[*] Training Classical Support Vector Machine (SVM)...")
    svm = SVC(kernel='rbf', C=1.0, class_weight='balanced', probability=True, random_state=42)
    svm.fit(X_train_scaled, y_train)

    # Validate model
    val_preds_svm = svm.predict(X_val_scaled)
    evaluate_performance(y_val, val_preds_svm, model_name="Classical SVM (Validation Set)")

    # Final Test evaluation
    test_preds_svm = svm.predict(X_test_scaled)
    test_probs_svm = svm.predict_proba(X_test_scaled)[:, 1]
    evaluate_performance(y_test, test_preds_svm, test_probs_svm, model_name="Classical SVM (Test Set)")

    # 5. Benchmark Quantum Support Vector Machine (QSVM)
    print("\n[*] Benchmark Quantum Support Vector Machine (QSVM)...")
    num_qubits = min(X_train_scaled.shape[1], 8) # Quantum circuit scaling

    feature_map = ZZFeatureMap(feature_dimension=num_qubits, reps=2)
    qkernel = FidelityQuantumKernel(feature_map=feature_map)
    qsvc = QSVC(quantum_kernel=qkernel)

    qsvc.fit(X_train_scaled[:, :num_qubits], y_train)

    test_preds_qsvm = qsvc.predict(X_test_scaled[:, :num_qubits])
    evaluate_performance(y_test, test_preds_qsvm, model_name="Quantum SVM Benchmark (Test Set)")

if __name__ == "__main__":
    main()

[*] Extracting SIFT features from TRAIN set...
[*] Extracting SIFT features from VAL set...
[*] Extracting SIFT features from TEST set...

[*] Training K-Means model on training descriptors (K = 50)...

[*] Training Classical Support Vector Machine (SVM)...

      CLASSICAL SVM (VALIDATION SET) EVALUATION      
Accuracy  : 100.00%
Precision : 100.00%
Recall    : 100.00%
F1-Score  : 100.00%

Confusion Matrix:
               Pred Normal  Pred CHD
Actual Normal  33          0
Actual CHD     0           28

      CLASSICAL SVM (TEST SET) EVALUATION      
Accuracy  : 100.00%
Precision : 100.00%
Recall    : 100.00%
F1-Score  : 100.00%

Confusion Matrix:
               Pred Normal  Pred CHD
Actual Normal  23          0
Actual CHD     0           20
ROC-AUC   : 1.0000

[*] Benchmark Quantum Support Vector Machine (QSVM)...


/tmp/ipykernel_1437/2255266642.py:207: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(feature_dimension=num_qubits, reps=2)



      QUANTUM SVM BENCHMARK (TEST SET) EVALUATION      
Accuracy  : 65.12%
Precision : 100.00%
Recall    : 25.00%
F1-Score  : 40.00%

Confusion Matrix:
               Pred Normal  Pred CHD
Actual Normal  23          0
Actual CHD     15          5


In [ ]:
import os
import glob
import cv2
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from qiskit_machine_learning.algorithms import PegasosQSVC
from skimage.segmentation import chan_vese
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

# Modern Qiskit Imports
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

# =========================================================
# STEP 1: Abstract-Compliant Preprocessing + ROI Bounding
# =========================================================
def preprocess_and_segment_roi(img_path):
    """
    Follows Bilateral Filter -> CLAHE -> Chan-Vese, but restricts
    Chan-Vese to an elliptical central mask to prevent background text leakage.
    """
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # 1. Bilateral Filter for noise removal
    denoised = cv2.bilateralFilter(img, d=7, sigmaColor=50, sigmaSpace=50)

    # 2. CLAHE for contrast enhancement
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)

    # 3. Restrict Chan-Vese to central cardiac zone (masks out machine text/borders)
    h, w = enhanced.shape
    mask_roi = np.zeros((h, w), dtype=np.uint8)
    center = (int(w / 2), int(h / 2))
    axes = (int(w * 0.38), int(h * 0.38))
    cv2.ellipse(mask_roi, center, axes, 0, 0, 360, 255, -1)

    focused_enhanced = cv2.bitwise_and(enhanced, enhanced, mask=mask_roi)

    # 4. Chan-Vese Active Contours (Higher mu=0.6 prevents noisy checkerboarding)
    img_norm = focused_enhanced / 255.0
    cv_mask = chan_vese(
        img_norm,
        mu=0.6,
        lambda1=1.0,
        lambda2=1.0,
        tol=1e-3,
        max_num_iter=100,
        dt=0.5
    )

    return (cv_mask * 255).astype(np.uint8)

# =========================================================
# STEP 2: SIFT Feature Extraction (Capped per Image)
# =========================================================
# Cap keypoints to top 150 to force learning only strong anatomical corners
sift = cv2.SIFT_create(nfeatures=150)

def load_dataset_features(dataset_split_dir):
    categories = ['Normal', 'CHD']
    image_desc_map, labels, all_descriptors = [], [], []

    for label_idx, cat in enumerate(categories):
        # Handle case-insensitive folder matching
        matched_folder = None
        for f in os.listdir(dataset_split_dir):
            if f.lower() == cat.lower():
                matched_folder = f
                break

        if not matched_folder:
            continue

        cat_path = os.path.join(dataset_split_dir, matched_folder)
        image_files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG']:
            image_files.extend(glob.glob(os.path.join(cat_path, ext)))

        for img_path in image_files:
            segmented_roi = preprocess_and_segment_roi(img_path)
            if segmented_roi is not None:
                _, desc = sift.detectAndCompute(segmented_roi, None)
                image_desc_map.append(desc)
                labels.append(label_idx)
                if desc is not None:
                    all_descriptors.append(desc)

    return image_desc_map, np.array(labels), all_descriptors

def build_bovw_histograms(image_desc_map, kmeans_model, num_clusters):
    X_bovw = []
    for desc in image_desc_map:
        histogram = np.zeros(num_clusters)
        if desc is not None and len(desc) > 0:
            words = kmeans_model.predict(desc)
            for word in words:
                histogram[word] += 1
            norm = np.linalg.norm(histogram)
            if norm > 0:
                histogram /= norm
        X_bovw.append(histogram)
    return np.array(X_bovw)

def evaluate_performance(y_true, y_pred, y_prob=None, model_name="Model"):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

    print(f"\n==========================================")
    print(f"      {model_name.upper()} EVALUATION      ")
    print(f"==========================================")
    print(f"Accuracy  : {acc * 100:.2f}%")
    print(f"Precision : {prec * 100:.2f}%")
    print(f"Recall    : {rec * 100:.2f}%")
    print(f"F1-Score  : {f1 * 100:.2f}%")

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print(f"               Pred Normal  Pred CHD")
    print(f"Actual Normal  {cm[0][0]:<11}  {cm[0][1] if cm.shape[1] > 1 else 0}")
    print(f"Actual CHD     {cm[1][0]:<11}  {cm[1][1] if cm.shape[1] > 1 else 0}")

    if y_prob is not None and len(np.unique(y_true)) > 1:
        auc = roc_auc_score(y_true, y_prob)
        print(f"ROC-AUC   : {auc:.4f}")


# =========================================================
# STEP 3: Model Pipeline Execution & Evaluation
# =========================================================
def run_improved_pipeline():
    DATASET_DIR = "./dataset_split"
    K_WORDS = 25
    NUM_QUBITS = 8

    print("[*] Extracting SIFT features from ROI-masked Chan-Vese images...")

    train_map, y_train, train_desc = load_dataset_features(os.path.join(DATASET_DIR, "train"))
    test_map, y_test, _ = load_dataset_features(os.path.join(DATASET_DIR, "test"))

    stacked_train = np.vstack([d for d in train_desc if d is not None])

    print(f"[*] Training K-Means visual vocabulary (K = {K_WORDS})...")
    kmeans = KMeans(n_clusters=K_WORDS, random_state=42, n_init=10)
    kmeans.fit(stacked_train)

    X_train = build_bovw_histograms(train_map, kmeans, K_WORDS)
    X_test = build_bovw_histograms(test_map, kmeans, K_WORDS)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- 1. Academically Rigorous Classical SVM Tuning ---
    print("\n[*] Tuning Classical SVM via GridSearchCV to find true generalization baseline...")
    param_grid = {
        'C': [0.1, 1, 10],
        'gamma': ['scale', 0.1, 0.01],
        'kernel': ['rbf', 'linear']
    }

    grid_search = GridSearchCV(
        SVC(class_weight='balanced', probability=True, random_state=42),
        param_grid,
        cv=5,
        n_jobs=-1
    )
    grid_search.fit(X_train_scaled, y_train)
    best_svm = grid_search.best_estimator_

    print(f" -> Best Classical Params Found: {grid_search.best_params_}")

    y_pred_svm = best_svm.predict(X_test_scaled)
    evaluate_performance(y_test, y_pred_svm, model_name="Tuned Classical SVM")


    print("\n==========================================")
    print("      CLASSICAL SVM REALISTIC PERFORMANCE ")
    print("==========================================")
    print(f"Accuracy  : {accuracy_score(y_test, y_pred_svm)*100:.2f}%")
    print(f"Precision : {precision_score(y_test, y_pred_svm, pos_label=1)*100:.2f}%")
    print(f"Recall    : {recall_score(y_test, y_pred_svm, pos_label=1)*100:.2f}%")
    print(f"F1-Score  : {f1_score(y_test, y_pred_svm, pos_label=1)*100:.2f}%")

    # --- 2. Enhanced Quantum SVM (Pegasos Algorithm + Phase Scaling) ---
    print("\n[*] Enhancing QSVM with Phase Scaling and Pegasos Algorithm...")

    # Step A: PCA to match Qubit count
    pca = PCA(n_components=NUM_QUBITS, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    # Step B: CRITICAL FIX - Scale PCA outputs to [0, pi] for quantum angle encoding
    q_scaler = MinMaxScaler(feature_range=(0, np.pi))
    X_train_q = q_scaler.fit_transform(X_train_pca)
    X_test_q = q_scaler.transform(X_test_pca)

    # Step C: Quantum Feature Map
    f_map = zz_feature_map(feature_dimension=NUM_QUBITS, reps=2)
    qkernel = FidelityQuantumKernel(feature_map=f_map)

   # Step D: Train Standard QSVC on a Balanced Subset with Phase-Scaled Data
    print("\n[*] Subsampling 60 balanced samples for rapid QSVC training...")

    # 1. Select 30 Normal and 30 CHD indices
    idx_normal = np.where(y_train == 0)[0][:30]
    idx_chd    = np.where(y_train == 1)[0][:30]
    balanced_idx = np.hstack([idx_normal, idx_chd])

    # 2. Subset the PHASE-SCALED data
    X_train_q_sub = X_train_q[balanced_idx]
    y_train_sub = y_train[balanced_idx]

    # 3. Train the standard QSVC (will take ~45 seconds)
    print(f"[*] Training QSVC on {len(y_train_sub)} samples...")
    qsvc = QSVC(quantum_kernel=qkernel)
    qsvc.fit(X_train_q_sub, y_train_sub)

    # 4. Predict on the phase-scaled test set
    y_pred_qsvm = qsvc.predict(X_test_q)
    evaluate_performance(y_test, y_pred_qsvm, model_name="Phase-Scaled Quantum SVM")

    # --- ADD THIS SUBSAMPLING FIX ---
    # Select 30 Normal and 30 CHD indices to speed up QSVM training
    idx_normal = np.where(y_train == 0)[0][:30]
    idx_chd    = np.where(y_train == 1)[0][:30]
    balanced_idx = np.hstack([idx_normal, idx_chd])

    # Fit with the balanced 60-sample subset
    qsvc = QSVC(quantum_kernel=qkernel) # Re-initialize QSVC to avoid retraining Pegasos
    qsvc.fit(X_train_pca[balanced_idx], y_train[balanced_idx])
    # ---------------------------------

    y_pred_qsvm = qsvc.predict(X_test_pca)

    print("\n==========================================")
    print("      QUANTUM SVM BENCHMARK PERFORMANCE   ")
    print("==========================================")
    print(f"Accuracy  : {accuracy_score(y_test, y_pred_qsvm)*100:.2f}%")
    print(f"Precision : {precision_score(y_test, y_pred_qsvm, pos_label=1)*100:.2f}%")
    print(f"Recall    : {recall_score(y_test, y_pred_qsvm, pos_label=1)*100:.2f}%")
    print(f"F1-Score  : {f1_score(y_test, y_pred_qsvm, pos_label=1)*100:.2f}%")

if __name__ == "__main__":
    run_improved_pipeline()

[*] Extracting SIFT features from ROI-masked Chan-Vese images...
[*] Training K-Means visual vocabulary (K = 25)...

[*] Tuning Classical SVM via GridSearchCV to find true generalization baseline...
 -> Best Classical Params Found: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}

      TUNED CLASSICAL SVM EVALUATION      
Accuracy  : 97.67%
Precision : 95.24%
Recall    : 100.00%
F1-Score  : 97.56%

Confusion Matrix:
               Pred Normal  Pred CHD
Actual Normal  22           1
Actual CHD     0            20

      CLASSICAL SVM REALISTIC PERFORMANCE 
Accuracy  : 97.67%
Precision : 95.24%
Recall    : 100.00%
F1-Score  : 97.56%

[*] Enhancing QSVM with Phase Scaling and Pegasos Algorithm...

[*] Subsampling 60 balanced samples for rapid QSVC training...
[*] Training QSVC on 60 samples...

      PHASE-SCALED QUANTUM SVM EVALUATION      
Accuracy  : 65.12%
Precision : 60.00%
Recall    : 75.00%
F1-Score  : 66.67%

Confusion Matrix:
               Pred Normal  Pred CHD
Actual Normal  13   

In [ ]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import chan_vese
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from google.colab import files
import ipywidgets as widgets
from IPython.display import display

# Qiskit for Quantum SVM Benchmark
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

# =========================================================
# STEP 1: Image Preprocessing & Chan-Vese Segmentation
# =========================================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import chan_vese

# =========================================================
# 1. Enhanced Preprocessing & Uniform Segmentation
# =========================================================
def preprocess_and_segment_4chamber(img_path_or_array, max_iter=100):
    """
    Standardizes external echo/X-ray inputs by isolating
    the 4-chamber cardiac region and applying smooth Chan-Vese contours.
    """
    if isinstance(img_path_or_array, str):
        img = cv2.imread(img_path_or_array, cv2.IMREAD_GRAYSCALE)
    else:
        img = img_path_or_array

    if img is None:
        return None, None

    # 1. Denoise and enhance contrast
    denoised = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)

    # 2. Dynamic Central ROI Mask (Isolates 4-chamber view & cuts peripheral text/fan borders)
    h, w = enhanced.shape
    roi_mask = np.zeros((h, w), dtype=np.uint8)
    center = (int(w / 2), int(h / 2))
    axes = (int(w * 0.40), int(h * 0.40))  # Focuses on central 80% chamber region
    cv2.ellipse(roi_mask, center, axes, 0, 0, 360, 255, -1)

    focused_img = cv2.bitwise_and(enhanced, enhanced, mask=roi_mask)

    # 3. Smooth Chan-Vese Segmentation
    # Higher 'mu' (0.6 - 0.8) prevents patchy fragmentation and yields continuous boundaries
    img_norm = focused_img / 255.0
    cv_mask = chan_vese(
        img_norm,
        mu=0.7,             # Smoothness penalty for clean chamber contours
        lambda1=1.0,
        lambda2=1.0,
        tol=1e-3,
        max_num_iter=max_iter
    )

    segmented_binary = (cv_mask * 255).astype(np.uint8)

    # 4. Extract Chamber Boundary Overlay
    contours, _ = cv2.findContours(segmented_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boundary_overlay = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.drawContours(boundary_overlay, contours, -1, (0, 255, 0), 2)  # Green chamber outlines

    return segmented_binary, boundary_overlay


# =========================================================
# STEP 2: SIFT Feature Extraction & Loader
# =========================================================
sift = cv2.SIFT_create(nfeatures=200)

def load_split_descriptors(split_dir):
    """Case-insensitive loader for dataset splits."""
    categories = ['Normal', 'CHD']
    image_desc_map = []
    labels = []
    all_descriptors = []

    existing_folders = os.listdir(split_dir)
    for label_idx, cat in enumerate(categories):
        target_folder = None
        for folder in existing_folders:
            if folder.lower() == cat.lower():
                target_folder = folder
                break

        if target_folder is None:
            continue

        cat_path = os.path.join(split_dir, target_folder)
        image_files = []
        for ext in ['*.png', '*.jpg', '*.jpeg', '*.PNG', '*.JPG', '*.JPEG', '*.BMP', '*.bmp']:
            image_files.extend(glob.glob(os.path.join(cat_path, ext)))

        for img_path in image_files:
            seg_img, _ = preprocess_and_segment_4chamber(img_path, max_iter=30) # Fixed function name and added return value handling
            if seg_img is not None:
                _, desc = sift.detectAndCompute(seg_img, None)
                image_desc_map.append(desc)
                labels.append(label_idx)
                if desc is not None:
                    all_descriptors.append(desc)

    return image_desc_map, np.array(labels), all_descriptors

# =========================================================
# STEP 3: Bag of Visual Words (BoVW) Feature Vectorization
# =========================================================
def build_bovw_histograms(image_desc_map, kmeans_model, num_clusters):
    X_bovw = []
    for desc in image_desc_map:
        histogram = np.zeros(num_clusters)
        if desc is not None and len(desc) > 0:
            words = kmeans_model.predict(desc)
            for word in words:
                histogram[word] += 1
            norm = np.linalg.norm(histogram)
            if norm > 0:
                histogram /= norm
        X_bovw.append(histogram)
    return np.array(X_bovw)

def evaluate_performance(y_true, y_pred, y_prob=None, model_name="Model"):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

    print(f"\n==========================================")
    print(f"      {model_name.upper()} EVALUATION      ")
    print(f"==========================================")
    print(f"Accuracy  : {acc * 100:.2f}%")
    print(f"Precision : {prec * 100:.2f}%")
    print(f"Recall    : {rec * 100:.2f}%")
    print(f"F1-Score  : {f1 * 100:.2f}%")

    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print(f"               Pred Normal  Pred CHD")
    print(f"Actual Normal  {cm[0][0]:<11}  {cm[0][1] if cm.shape[1] > 1 else 0}")
    print(f"Actual CHD     {cm[1][0]:<11}  {cm[1][1] if cm.shape[1] > 1 else 0}")

    if y_prob is not None and len(np.unique(y_true)) > 1:
        auc = roc_auc_score(y_true, y_prob)
        print(f"ROC-AUC   : {auc:.4f}")
    # =========================================================
# STEP 4: Model Pipeline Execution
# =========================================================
def run_pipeline():
    DATASET_DIR = "./dataset_split"
    K_WORDS = 25
    NUM_QUBITS = 8

    print("[*] Extracting SIFT features from 4-chamber Chan-Vese images...")

    # Load and process datasets using the newly defined 4-chamber segmentation
    train_map, y_train, train_desc = load_split_descriptors(os.path.join(DATASET_DIR, "train"))
    test_map, y_test, _ = load_split_descriptors(os.path.join(DATASET_DIR, "test"))

    # Check if descriptors were extracted successfully
    if not train_desc:
        print("\nCRITICAL ERROR: No SIFT descriptors were extracted from the training set.")
        return

    stacked_train = np.vstack([d for d in train_desc if d is not None])

    # 1. Build K-Means Visual Vocabulary
    print(f"\n[*] Training K-Means visual vocabulary (K = {K_WORDS})...")
    kmeans = KMeans(n_clusters=K_WORDS, random_state=42, n_init=10)
    kmeans.fit(stacked_train)

    # 2. Transform splits into BoVW histograms
    X_train = build_bovw_histograms(train_map, kmeans, K_WORDS)
    X_test = build_bovw_histograms(test_map, kmeans, K_WORDS)

    # 3. Feature Scaling for Classical SVM
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ---------------------------------------------------------
    # 4. Train Classical Support Vector Machine (SVM)
    # ---------------------------------------------------------
    print("\n[*] Tuning Classical SVM via GridSearchCV...")
    param_grid = {
        'C': [0.1, 1, 10],
        'gamma': ['scale', 0.1, 0.01],
        'kernel': ['rbf', 'linear']
    }

    grid_search = GridSearchCV(
        SVC(class_weight='balanced', probability=True, random_state=42),
        param_grid,
        cv=5,
        n_jobs=-1
    )
    grid_search.fit(X_train_scaled, y_train)
    best_svm = grid_search.best_estimator_

    print(f" -> Best Classical Params Found: {grid_search.best_params_}")

    y_pred_svm = best_svm.predict(X_test_scaled)
    y_prob_svm = best_svm.predict_proba(X_test_scaled)[:, 1] if len(np.unique(y_train)) > 1 else None
    evaluate_performance(y_test, y_pred_svm, y_prob_svm, model_name="Tuned Classical SVM")

    # ---------------------------------------------------------
    # 5. Train Quantum Support Vector Machine (QSVM)
    # ---------------------------------------------------------
    print("\n[*] Enhancing QSVM with Phase Scaling and Pegasos Algorithm...")

    # Step A: PCA to match Qubit count
    pca = PCA(n_components=NUM_QUBITS, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)

    # Step B: Scale PCA outputs to [0, pi] for quantum angle encoding
    q_scaler = MinMaxScaler(feature_range=(0, np.pi))
    X_train_q = q_scaler.fit_transform(X_train_pca)
    X_test_q = q_scaler.transform(X_test_pca)

    # Step C: Quantum Feature Map
    f_map = zz_feature_map(feature_dimension=NUM_QUBITS, reps=2)
    qkernel = FidelityQuantumKernel(feature_map=f_map)

    # Step D: Train QSVC on a Balanced Subset for Speed
    print("[*] Subsampling 60 balanced samples for rapid QSVC training...")
    idx_normal = np.where(y_train == 0)[0][:30]
    idx_chd    = np.where(y_train == 1)[0][:30]
    balanced_idx = np.hstack([idx_normal, idx_chd])

    X_train_q_sub = X_train_q[balanced_idx]
    y_train_sub = y_train[balanced_idx]

    print(f"[*] Training QSVC on {len(y_train_sub)} samples...")
    qsvc = QSVC(quantum_kernel=qkernel)
    qsvc.fit(X_train_q_sub, y_train_sub)

    y_pred_qsvm = qsvc.predict(X_test_q)
    evaluate_performance(y_test, y_pred_qsvm, model_name="Phase-Scaled Quantum SVM")

    # Return dictionary of models/variables to be used globally in your Paper Proofs cell
    return {
        'kmeans': kmeans,
        'scaler': scaler,
        'svm': best_svm,
        'qsvc': qsvc,
        'pca': pca,
        'q_scaler': q_scaler,
        'X_test_scaled': X_test_scaled,
        'y_test': y_test
    }

# =========================================================
# STEP 5: Execution Pipeline
# =========================================================
if __name__ == "__main__":
    globals_pipeline = run_pipeline()

[*] Extracting SIFT features from 4-chamber Chan-Vese images...

[*] Training K-Means visual vocabulary (K = 25)...

[*] Tuning Classical SVM via GridSearchCV...
 -> Best Classical Params Found: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}

      TUNED CLASSICAL SVM EVALUATION      
Accuracy  : 100.00%
Precision : 100.00%
Recall    : 100.00%
F1-Score  : 100.00%

Confusion Matrix:
               Pred Normal  Pred CHD
Actual Normal  23           0
Actual CHD     0            20
ROC-AUC   : 1.0000

[*] Enhancing QSVM with Phase Scaling and Pegasos Algorithm...
[*] Subsampling 60 balanced samples for rapid QSVC training...
[*] Training QSVC on 60 samples...

      PHASE-SCALED QUANTUM SVM EVALUATION      
Accuracy  : 76.74%
Precision : 77.78%
Recall    : 70.00%
F1-Score  : 73.68%

Confusion Matrix:
               Pred Normal  Pred CHD
Actual Normal  19           4
Actual CHD     6            14


In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# Set style for academic publication figures
plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})
os.makedirs('./paper_proofs', exist_ok=True)

# ---------------------------------------------------------
# 1. Pipeline Proof: Preprocessing & Segmentation Stages
# ---------------------------------------------------------
print("[*] Generating Pipeline Visual Proof (Figure 1)...")

# Grab one sample from Normal and one from CHD
sample_normal = glob.glob('./dataset_split/test/Normal/*.*')[0]
sample_chd = glob.glob('./dataset_split/test/CHD/*.*')[0]

def generate_pipeline_figure(img_path, title, save_filename):
    raw_img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

    # Preprocessing stages
    denoised = cv2.bilateralFilter(raw_img, d=9, sigmaColor=75, sigmaSpace=75)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(denoised)
    segmented = preprocess_and_segment(raw_img, max_iter=30)

    # Extract SIFT keypoints on segmented image
    keypoints, _ = sift.detectAndCompute(segmented, None)
    sift_overlay = cv2.drawKeypoints(
        segmented, keypoints, None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    )

    # Plot 4-stage comparison
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(raw_img, cmap='gray')
    axes[0].set_title("Raw Input X-Ray")
    axes[0].axis('off')

    axes[1].imshow(enhanced, cmap='gray')
    axes[1].set_title("Bilateral + CLAHE")
    axes[1].axis('off')

    axes[2].imshow(segmented, cmap='gray')
    axes[2].set_title("Chan-Vese Contour")
    axes[2].axis('off')

    axes[3].imshow(sift_overlay)
    axes[3].set_title(f"SIFT Features ({len(keypoints)} pts)")
    axes[3].axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'./paper_proofs/{save_filename}.png', dpi=300, bbox_inches='tight')
    plt.show()

generate_pipeline_figure(sample_normal, "Normal Controls Preprocessing Pipeline", "fig1_pipeline_normal")
generate_pipeline_figure(sample_chd, "CHD Preprocessing Pipeline", "fig1_pipeline_chd")


# ---------------------------------------------------------
# 2. Feature Proof: Bag of Visual Words (BoVW) Histograms
# ---------------------------------------------------------
print("[*] Generating BoVW Feature Histogram Proof (Figure 2)...")

kmeans = globals_pipeline['kmeans']
scaler = globals_pipeline['scaler']

# Compute mean visual word distributions across classes
def get_class_mean_histogram(split_dir, cat_name):
    cat_path = os.path.join(split_dir, cat_name)
    img_files = glob.glob(os.path.join(cat_path, '*.*'))
    histograms = []

    for p in img_files:
        seg = preprocess_and_segment(p, max_iter=30)
        if seg is not None:
            _, desc = sift.detectAndCompute(seg, None)
            if desc is not None and len(desc) > 0:
                h = np.zeros(50)
                words = kmeans.predict(desc)
                for w in words:
                    h[w] += 1
                norm = np.linalg.norm(h)
                if norm > 0:
                    h /= norm
                histograms.append(h)
    return np.mean(histograms, axis=0)

mean_hist_normal = get_class_mean_histogram('./dataset_split/test', 'Normal')
mean_hist_chd = get_class_mean_histogram('./dataset_split/test', 'CHD')

plt.figure(figsize=(12, 4))
x = np.arange(50)
plt.bar(x - 0.2, mean_hist_normal, width=0.4, label='Normal Control', color='#2b5c8f', alpha=0.85)
plt.bar(x + 0.2, mean_hist_chd, width=0.4, label='CHD', color='#d95f02', alpha=0.85)
plt.xlabel("Visual Word Cluster ID (K = 50)", fontweight='bold')
plt.ylabel("Mean Normalized Frequency", fontweight='bold')
plt.title("BoVW Visual Vocabulary Feature Distribution Across Classes", fontweight='bold')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig('./paper_proofs/fig2_bovw_distribution.png', dpi=300, bbox_inches='tight')
plt.show()


# ---------------------------------------------------------
# 3. Model Benchmark Proof: Confusion Matrices
# ---------------------------------------------------------
print("[*] Generating Confusion Matrices Proof (Figure 3)...")

svm = globals_pipeline['svm']
qsvc = globals_pipeline['qsvc']
pca = globals_pipeline['pca']

# Predict test set
cm_svm = confusion_matrix(y_test, svm.predict(X_test_scaled))
cm_qsvm = confusion_matrix(y_test, qsvc.predict(pca.transform(X_test_scaled)))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
labels = ['Normal', 'CHD']

sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=labels, yticklabels=labels)
axes[0].set_title("Classical SVM (Test Set)", fontweight='bold')
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(cm_qsvm, annot=True, fmt='d', cmap='Oranges', ax=axes[1], xticklabels=labels, yticklabels=labels)
axes[1].set_title("Quantum SVM Benchmark (Test Set)", fontweight='bold')
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.savefig('./paper_proofs/fig3_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()


# ---------------------------------------------------------
# 4. Tabular Proof: Export Quantitative Results CSV
# ---------------------------------------------------------
print("[*] Exporting Quantitative Results Table (Table 1)...")

results_data = {
    'Model Architecture': ['Classical Support Vector Machine (RBF)', 'Quantum Support Vector Machine (QSVC)'],
    'Feature Vector': ['50-dim BoVW Histogram', '8 Qubit PCA Features'],
    'Accuracy (%)': [
        accuracy_score(y_test, svm.predict(X_test_scaled)) * 100,
        accuracy_score(y_test, qsvc.predict(pca.transform(X_test_scaled))) * 100
    ],
    'Precision (%)': [
        precision_score(y_test, svm.predict(X_test_scaled), pos_label=1) * 100,
        precision_score(y_test, qsvc.predict(pca.transform(X_test_scaled)), pos_label=1, zero_division=0) * 100
    ],
    'Recall (%)': [
        recall_score(y_test, svm.predict(X_test_scaled), pos_label=1) * 100,
        recall_score(y_test, qsvc.predict(pca.transform(X_test_scaled)), pos_label=1, zero_division=0) * 100
    ],
    'F1-Score (%)': [
        f1_score(y_test, svm.predict(X_test_scaled), pos_label=1) * 100,
        f1_score(y_test, qsvc.predict(pca.transform(X_test_scaled)), pos_label=1, zero_division=0) * 100
    ]
}

df_results = pd.DataFrame(results_data)
df_results.to_csv('./paper_proofs/table1_model_performance.csv', index=False)
print("\n[+] Table 1 Saved to CSV:\n")
print(df_results.to_string(index=False))

print("\n[+] All proof artifacts saved under directory: ./paper_proofs")

[*] Generating Pipeline Visual Proof (Figure 1)...


NameError: name 'preprocess_and_segment' is not defined

In [ ]:
python train_and_evaluate.py